# Cyclistic Bike-Share Case Study

## Business Task
Analyze how annual members and casual riders use Cyclistic bikes differently and provide recommendations to convert casual riders into annual members.

## Data Source
The dataset used for this analysis is the Cyclistic Bike-Share dataset provided by Divvy and made available through Google Data Analytics Capstone resources.

In [1]:
import pandas as pd
import numpy as np
import os
import glob

In [2]:
all_files = glob.glob(r"D:\capstone project cyclistic dataset\Raw_dataset\*.csv") #Read all csv files
# Empty list
df_list = []

# Read files one by one
for file in all_files:
    df = pd.read_csv(file)
    df_list.append(df)

# Combine all datasets
cyclistic_data = pd.concat(df_list, ignore_index=True)

# Show first 5 rows
cyclistic_data.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,AF3863596DF9D94B,classic_bike,2025-04-27 14:29:34.619,2025-04-27 14:36:23.584,Troy St & Elston Ave,15631,Richmond St & Diversey Ave,15645,41.945244,-87.706650,41.931902,-87.701195,member
1,8B38081EBE918800,electric_bike,2025-04-23 17:48:51.863,2025-04-23 17:59:06.015,Wabash Ave & Adams St,KA1503000015,Green St & Madison St,TA1307000120,41.879472,-87.625689,41.881859,-87.649264,member
2,1C7F1DE826BBBC8D,electric_bike,2025-04-05 17:55:30.845,2025-04-05 18:05:40.032,Damen Ave & Cortland St,13133,California Ave & Fletcher St,15642,41.915983,-87.677335,41.938429,-87.698008,member
3,CAD23D69A79A6C3B,classic_bike,2025-04-03 08:22:04.493,2025-04-03 08:32:06.099,Clark St & Elm St,TA1307000039,Orleans St & Merchandise Mart Plaza,TA1305000022,41.902973,-87.631280,41.888243,-87.636390,member
4,BE241E601482E0AB,electric_bike,2025-04-15 06:09:55.293,2025-04-15 06:19:58.942,Western Ave & Walton St,KA1504000103,Damen Ave & Charleston St,13288,41.898418,-87.686596,41.920082,-87.677855,member


## Data Cleaning and Preparation
The dataset was cleaned by checking for missing values, removing invalid records, converting date columns to datetime format, creating new features, and preparing the data for analysis.

In [3]:
cyclistic_data.shape

(5931009, 13)

In [4]:
cyclistic_data.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')

In [5]:
cyclistic_data.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1262388
start_station_id      1262388
end_station_name      1325903
end_station_id        1325903
start_lat                   0
start_lng                   0
end_lat                  5950
end_lng                  5950
member_casual               0
dtype: int64

In [6]:
cyclistic_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5931009 entries, 0 to 5931008
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 588.3+ MB


In [7]:
cyclistic_data['started_at'] = pd.to_datetime(cyclistic_data['started_at'])  #covert date columns to date time

cyclistic_data['ended_at'] = pd.to_datetime(cyclistic_data['ended_at'])

In [8]:
# Create ride length column
cyclistic_data['ride_length'] = (
    cyclistic_data['ended_at'] - cyclistic_data['started_at']
).dt.total_seconds() / 60

cyclistic_data[['started_at', 'ended_at', 'ride_length']].head()

,started_at,ended_at,ride_length
0,2025-04-27 14:29:34.619,2025-04-27 14:36:23.584,6.816083
1,2025-04-23 17:48:51.863,2025-04-23 17:59:06.015,10.235867
2,2025-04-05 17:55:30.845,2025-04-05 18:05:40.032,10.153117
3,2025-04-03 08:22:04.493,2025-04-03 08:32:06.099,10.026767
4,2025-04-15 06:09:55.293,2025-04-15 06:19:58.942,10.060817


In [9]:
# Remove negative and zero ride lengths
cyclistic_data = cyclistic_data[cyclistic_data['ride_length'] > 0]

In [10]:
# Check dataset shape again
cyclistic_data.shape

(5930980, 14)

In [11]:
# Create new date columns
cyclistic_data['date'] = cyclistic_data['started_at'].dt.date

cyclistic_data['month'] = cyclistic_data['started_at'].dt.month_name()

cyclistic_data['day'] = cyclistic_data['started_at'].dt.day_name()

cyclistic_data['hour'] = cyclistic_data['started_at'].dt.hour

In [12]:
cyclistic_data[['started_at','month','day','hour']].head()

,started_at,month,day,hour
0,2025-04-27 14:29:34.619,April,Sunday,14
1,2025-04-23 17:48:51.863,April,Wednesday,17
2,2025-04-05 17:55:30.845,April,Saturday,17
3,2025-04-03 08:22:04.493,April,Thursday,8
4,2025-04-15 06:09:55.293,April,Tuesday,6


In [13]:
cyclistic_data.groupby('member_casual')['ride_length'].mean()

member_casual
casual    22.454279
member    12.351198
Name: ride_length, dtype: float64

In [14]:
cyclistic_data.groupby('day')['ride_length'].mean().sort_values(ascending=False)

day
Sunday       19.401124
Saturday     19.059395
Friday       15.981405
Monday       15.262629
Thursday     14.366157
Tuesday      14.071634
Wednesday    13.789778
Name: ride_length, dtype: float64

In [15]:
cyclistic_data['member_casual'].value_counts()

member_casual
member    3808725
casual    2122255
Name: count, dtype: int64

In [16]:
cyclistic_data.groupby('member_casual')['ride_id'].count()

member_casual
casual    2122255
member    3808725
Name: ride_id, dtype: int64

In [17]:
cyclistic_data.groupby('rideable_type')['ride_id'].count()

rideable_type
classic_bike     2030972
electric_bike    3900008
Name: ride_id, dtype: int64

In [18]:
cyclistic_data.groupby('rideable_type')['ride_length'].mean()

rideable_type
classic_bike     23.381279
electric_bike    12.104924
Name: ride_length, dtype: float64

In [19]:
cyclistic_data.to_csv(
    r"D:\capstone project cyclistic dataset\Cleaned_data\cyclistic_cleaned_data.csv",
    index=False)

print("Cleaned dataset saved successfully")

Cleaned dataset saved successfully


In [20]:
import pandas as pd
cyclistic_data = pd.read_csv(
    r"D:\capstone project cyclistic dataset\Cleaned_data\cyclistic_cleaned_data.csv"
)

In [21]:
cyclistic_data.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,date,month,day,hour
0,AF3863596DF9D94B,classic_bike,2025-04-27 14:29:34.619,2025-04-27 14:36:23.584,Troy St & Elston Ave,15631,Richmond St & Diversey Ave,15645,41.945244,-87.706650,41.931902,-87.701195,member,6.816083,2025-04-27,April,Sunday,14
1,8B38081EBE918800,electric_bike,2025-04-23 17:48:51.863,2025-04-23 17:59:06.015,Wabash Ave & Adams St,KA1503000015,Green St & Madison St,TA1307000120,41.879472,-87.625689,41.881859,-87.649264,member,10.235867,2025-04-23,April,Wednesday,17
2,1C7F1DE826BBBC8D,electric_bike,2025-04-05 17:55:30.845,2025-04-05 18:05:40.032,Damen Ave & Cortland St,13133,California Ave & Fletcher St,15642,41.915983,-87.677335,41.938429,-87.698008,member,10.153117,2025-04-05,April,Saturday,17
3,CAD23D69A79A6C3B,classic_bike,2025-04-03 08:22:04.493,2025-04-03 08:32:06.099,Clark St & Elm St,TA1307000039,Orleans St & Merchandise Mart Plaza,TA1305000022,41.902973,-87.631280,41.888243,-87.636390,member,10.026767,2025-04-03,April,Thursday,8
4,BE241E601482E0AB,electric_bike,2025-04-15 06:09:55.293,2025-04-15 06:19:58.942,Western Ave & Walton St,KA1504000103,Damen Ave & Charleston St,13288,41.898418,-87.686596,41.920082,-87.677855,member,10.060817,2025-04-15,April,Tuesday,6


## Exploratory Data Analysis (EDA)
This section explores riding patterns, ride duration, bike types, monthly trends, and differences between casual riders and annual members.

# Ride Count by Month

In [22]:
cyclistic_data.groupby('month')['ride_id'].count().sort_values(ascending=False)

month
April        819628
August       790317
July         763458
September    714562
June         678801
October      646096
May          502615
November     356483
March        317026
February     201454
December     140522
January          18
Name: ride_id, dtype: int64

# Ride Count by Hour

In [23]:
cyclistic_data.groupby('hour')['ride_id'].count().sort_values(ascending=False)

hour
17    618170
16    550065
18    494682
15    417972
8     352277
19    349924
14    348208
12    338505
13    337555
11    293483
7     266432
9     252451
20    245548
10    242498
21    198163
22    152701
6     135304
23    101031
0      75165
5      48512
1      47167
2      30027
3      18029
4      17111
Name: ride_id, dtype: int64

# Member vs Bike Type

In [24]:
cyclistic_data.groupby(['member_casual','rideable_type'])['ride_id'].count()

member_casual  rideable_type
casual         classic_bike      696709
               electric_bike    1425546
member         classic_bike     1334263
               electric_bike    2474462
Name: ride_id, dtype: int64

# Member vs Day

In [25]:
cyclistic_data.groupby(['member_casual','day'])['ride_id'].count()

member_casual  day      
casual         Friday       332368
               Monday       244094
               Saturday     432569
               Sunday       354677
               Thursday     278174
               Tuesday      241068
               Wednesday    239305
member         Friday       557529
               Monday       536380
               Saturday     472177
               Sunday       408538
               Thursday     622507
               Tuesday      609785
               Wednesday    601809
Name: ride_id, dtype: int64

# Save Dataset before outlier removal

In [37]:
cyclistic_data.to_csv(
    r"D:\capstone project cyclistic dataset\Cleaned_data\cyclistic_final_data.csv",
    index=False
)

## Outlier Detection and Removal

In [38]:
cyclistic_data.groupby('month')['ride_length'].mean()

month
April        12.449627
August       15.722427
December     10.485593
February     10.870951
January      13.826110
July         15.343560
June         15.481799
March        12.085453
May          14.135165
November     11.459748
October      13.062222
September    14.310523
Name: ride_length, dtype: float64

In [28]:
cyclistic_data.groupby('month')['ride_length'].max()

month
April        1499.971583
August       1499.971150
December     1499.961817
February     1499.961383
January      1499.943883
July         1500.896883
June         1574.900183
March        1559.950183
May          1499.980583
November     1499.973167
October      1499.965117
September    1499.976000
Name: ride_length, dtype: float64

In [29]:
cyclistic_data[
    (cyclistic_data['month'] == 'January') &
    (cyclistic_data['ride_length'] > 300)
]['ride_length'].count()

np.int64(6)

In [30]:
cyclistic_data[
    (cyclistic_data['month'] == 'January')
]['ride_length'].describe()

count      18.000000
mean      500.422909
std       708.977379
min         3.626067
25%        12.152946
50%        15.641442
75%      1460.334242
max      1499.943883
Name: ride_length, dtype: float64

In [31]:
cyclistic_data = cyclistic_data[
    (cyclistic_data['ride_length'] > 0) &
    (cyclistic_data['ride_length'] <= 300)
]

In [32]:
cyclistic_data.groupby('month')['ride_length'].mean()

month
April        12.449627
August       15.722427
December     10.485593
February     10.870951
January      13.826110
July         15.343560
June         15.481799
March        12.085453
May          14.135165
November     11.459748
October      13.062222
September    14.310523
Name: ride_length, dtype: float64

# Final Dataset After Removing Outlier

In [36]:
cyclistic_data.to_csv(
    r"D:\capstone project cyclistic dataset\Cleaned_data\cyclistic_final_data_v2.csv",
    index=False
)

print("Saved")

Saved


## Key Findings

- Members completed more rides than casual riders.
- Casual riders had longer average ride durations.
- Ride activity peaked during summer months.
- Weekend rides were generally longer than weekday rides.
- Electric bikes were used more frequently than classic bikes.

## Recommendations

- Promote annual memberships to frequent casual riders.
- Offer weekend membership promotions.
- Target casual riders during peak summer months.
- Highlight membership benefits through marketing campaigns.